# Training JEV networks from Qwen

This is the notebook I wrote live on stream while working out the core trick behind training a JEV network on top of an existing autoregressive LM (Qwen3-0.6B).

**Livestream:** https://youtube.com/live/AzxoU7kxjig

**The goal:** give the LM a question and several options in *one* sequence, and get one embedding per option out of a single forward pass. The catch is that a normal LM reads left to right, so the order you list the options in changes what it sees. We want the model to be **choice-order invariant**: shuffle the options, and each option's embedding stays exactly the same.

Here's how the notebook gets there:

1. Tokenize the question and each choice separately, then stitch the tokens together, so we know the exact span of every choice.
2. Manipulate the position ids so every choice starts at the same position, right after the question.
3. Build a custom attention mask so each choice only sees the question and its own tokens (causally), never the other choices. The answer tokens at the end see everything.
4. Run Qwen with these, read one embedding per choice plus one answer embedding, and check that swapping the choices just swaps their embeddings. It does!
5. Sketch a tiny self-attention head over the choice and answer embeddings, as a first idea for the output layer.

The notebook is a demo, so it stops there. Batching, the loss, LoRA adapters, the improved head, and the actual training and evaluation live in the Python files in this repo (`tokenization.py`, `dataset.py`, `network.py`, `train.py`, `inference.py`). The functions you see here carry over into `tokenization.py` almost unchanged.


In [ ]:
import torch
from transformers import AutoTokenizer, Qwen3Model
model_name = "Qwen/Qwen3-0.6B" # SDPA style

model = Qwen3Model.from_pretrained(model_name, dtype=torch.float32) # fp32 so we dont get into weird precision issues
tokenizer = AutoTokenizer.from_pretrained(model_name)


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 916.57it/s] 
[transformers] Qwen3Model LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [169]:
model

Qwen3Model(
  (embed_tokens): Embedding(151936, 1024)
  (layers): ModuleList(
    (0-27): 28 x Qwen3DecoderLayer(
      (self_attn): Qwen3Attention(
        (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
        (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
        (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
        (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
        (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
      )
      (mlp): Qwen3MLP(
        (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
        (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
      (post_attention_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
    )
  )
  (norm): Qwen3RM

### Tokenization can work by forming strings like these

In [171]:
question = "What is the capital of India?"
choices = ["New Delhi", "Mumbai", "Kolkata"]

input_prompt = """What is the capital of India?
<option> Mumbai </option> </option>
<option> New Delhi </option>
<option> Kolkata </option>

Answer is: [ANSWER]
"""

input_ids = tokenizer.encode(input_prompt)

print(input_ids)

[3838, 374, 279, 6722, 315, 6747, 5267, 14296, 29, 34712, 690, 2047, 29, 690, 2047, 397, 14296, 29, 1532, 21996, 690, 2047, 397, 14296, 29, 81534, 690, 2047, 1339, 16141, 374, 25, 508, 11692, 39351, 921]


### But a better way is to tokenize first -> and then coalesce it, so you know the exact choice boundaries and spans

In [173]:

def tokenize_prompt(question,choices):
    choice_begin_token = tokenizer.encode("<option>")
    choice_end_token = tokenizer.encode("</option>")
    answer_prompt = "Answer is: [ANSWER]"

    question_tokens = tokenizer.encode(question)
    answer_tokens = tokenizer.encode(answer_prompt)

    choice_tokens = [
        tokenizer.encode(choice) for choice in choices
    ]

    final_tokens = []
    final_tokens.extend(question_tokens)

    span = {
        "question_end_idx": len(question_tokens) - 1,
        "choices_end_idx": [],
        "answer_end_idx": 0
    }

    for choice in choice_tokens:
        final_tokens.extend(choice_begin_token)
        final_tokens.extend(choice)
        final_tokens.extend(choice_end_token)
        span["choices_end_idx"].append(len(final_tokens) - 1)

    final_tokens.extend(answer_tokens)
    span["answer_end_idx"] = len(final_tokens) - 1

    return final_tokens, span

question = "What is the capital of India - please tell me?"
c1 = "New Delhi" 
c2 = "Mumbai"
c3 = "Kolkata"
final_tokens, span = tokenize_prompt(question, [c1, c2])

print(final_tokens)
print(span)

[3838, 374, 279, 6722, 315, 6747, 481, 4486, 3291, 752, 30, 14296, 29, 3564, 21996, 522, 2047, 29, 14296, 29, 44, 29855, 522, 2047, 29, 16141, 374, 25, 508, 11692, 39351, 60]
{'question_end_idx': 10, 'choices_end_idx': [17, 24], 'answer_end_idx': 31}


### We are manipulating position ids here, such that each choice token begins after the question finishes.

Instead of 

0 1 2 3 4 | 5 6 7 | 8 9 10 11 | 12 13 14 ...

we do:

0 1 2 3 4 | 5 6 7 | 5 6 7 8 | 5 6 7


where | indicates choice boundaries

![Position ids: the question and each choice are tokenized separately, and every choice restarts its positions right after the question](assets/pos_idxs.png)


In [176]:
def generate_position_ids(span):
    position_ids = []

    start_idx = 0
    for i in range(span["question_end_idx"] + 1):
        position_ids.append(start_idx)
        start_idx += 1

    next_choice_begin_idx = start_idx

    for choice_end_idx in span["choices_end_idx"]:
        for idx in range(choice_end_idx - next_choice_begin_idx + 1):
            position_ids.append(start_idx + idx)

        next_choice_begin_idx = choice_end_idx + 1

    for idx in range(span["answer_end_idx"] + 1 - len(position_ids)):
        position_ids.append(start_idx + idx)

    return position_ids


position_ids = generate_position_ids(span)

print(len(position_ids))

for pos_idx, token_idx in zip(position_ids, final_tokens):
    print(f"{pos_idx} | {token_idx} | {tokenizer.decode(token_idx)}")

32
0 | 3838 | What
1 | 374 |  is
2 | 279 |  the
3 | 6722 |  capital
4 | 315 |  of
5 | 6747 |  India
6 | 481 |  -
7 | 4486 |  please
8 | 3291 |  tell
9 | 752 |  me
10 | 30 | ?
11 | 14296 | <option
12 | 29 | >
13 | 3564 | New
14 | 21996 |  Delhi
15 | 522 | </
16 | 2047 | option
17 | 29 | >
11 | 14296 | <option
12 | 29 | >
13 | 44 | M
14 | 29855 | umbai
15 | 522 | </
16 | 2047 | option
17 | 29 | >
11 | 16141 | Answer
12 | 374 |  is
13 | 25 | :
14 | 508 |  [
15 | 11692 | ANS
16 | 39351 | WER
17 | 60 | ]


### Manipulating the attention masks as well, so each choice only sees its own tokens causally and the question tokens. No choice tokens can see each other, at this stage.


![Attention mask: each choice sees the state and itself, never the other choices, while the answer token sees everything](assets/masking.png)

In [177]:
import torch
def get_attention_mask(span):
    L = span["answer_end_idx"] + 1
    attention_mask = torch.zeros((L, L))

    question_len = span["question_end_idx"] + 1

    # Question tokens look back at everything in the past (within the question)
    attention_mask[:question_len, :question_len] = torch.tril(torch.ones(question_len, question_len))

    # Data structure that stores the spans of the choices (begin, end of each choices)
    choice_spans = []
    choice_start_idx = question_len
    for choice_end_idx in span["choices_end_idx"]:
        choice_spans.append((choice_start_idx, choice_end_idx))
        choice_start_idx = choice_end_idx + 1


    # Looped over each span of the choices and set the attention mask for the span
    for choice_start_idx, choice_end_idx in choice_spans:

        
        # Each choice tokens looks back within that choice span (using causal mask)
        attention_mask[
            choice_start_idx:choice_end_idx + 1, 
            choice_start_idx:choice_end_idx + 1
        ] = torch.tril(torch.ones(choice_end_idx - choice_start_idx + 1, 
                                choice_end_idx - choice_start_idx + 1)
                        )

    # Choice tokens look at question tokens
    attention_mask[question_len:, :question_len] = 1

    # Answer tokens looks back at everything in the past
    answer_tokens_begin = choice_spans[-1][1] + 1
    attention_mask[answer_tokens_begin:, :answer_tokens_begin] = 1
    attention_mask[answer_tokens_begin:, answer_tokens_begin:] = torch.tril(torch.ones(L - answer_tokens_begin, L - answer_tokens_begin))
    return attention_mask

am = get_attention_mask(span)

for idx in range(am.shape[0]):
    print(f"{idx:2d} |", "|".join(
        ["X" if x == "1" else " " for x in am[idx].numpy().astype(int).astype(str)]
    ))

 0 | X| | | | | | | | | | | | | | | | | | | | | | | | | | | | | | | 
 1 | X|X| | | | | | | | | | | | | | | | | | | | | | | | | | | | | | 
 2 | X|X|X| | | | | | | | | | | | | | | | | | | | | | | | | | | | | 
 3 | X|X|X|X| | | | | | | | | | | | | | | | | | | | | | | | | | | | 
 4 | X|X|X|X|X| | | | | | | | | | | | | | | | | | | | | | | | | | | 
 5 | X|X|X|X|X|X| | | | | | | | | | | | | | | | | | | | | | | | | | 
 6 | X|X|X|X|X|X|X| | | | | | | | | | | | | | | | | | | | | | | | | 
 7 | X|X|X|X|X|X|X|X| | | | | | | | | | | | | | | | | | | | | | | | 
 8 | X|X|X|X|X|X|X|X|X| | | | | | | | | | | | | | | | | | | | | | | 
 9 | X|X|X|X|X|X|X|X|X|X| | | | | | | | | | | | | | | | | | | | | | 
10 | X|X|X|X|X|X|X|X|X|X|X| | | | | | | | | | | | | | | | | | | | | 
11 | X|X|X|X|X|X|X|X|X|X|X|X| | | | | | | | | | | | | | | | | | | | 
12 | X|X|X|X|X|X|X|X|X|X|X|X|X| | | | | | | | | | | | | | | | | | | 
13 | X|X|X|X|X|X|X|X|X|X|X|X|X|X| | | | | | | | | | | | | | | | | | 
14 | X|X|X|X|X|X|X|X|X|X|X|X|X|X|X

## Wrapping it all together

In [178]:

def forward(question, choices):
    final_tokens, span = tokenize_prompt(question, choices)

    position_ids = generate_position_ids(span)
    final_tokens = torch.tensor(final_tokens).unsqueeze(0)
    position_ids = torch.tensor(position_ids).unsqueeze(0)
    attention_mask = get_attention_mask(span)
    attention_mask = attention_mask.unsqueeze(0).unsqueeze(0)
    attention_mask = torch.where(attention_mask == 0, -float('inf'), 0) 
    output = model(
        input_ids=final_tokens,
        attention_mask=attention_mask,
        position_ids=position_ids,
        past_key_values=None,
        inputs_embeds=None,
        use_cache=False
    )
    choice_embeddings = torch.stack([output.last_hidden_state[0][choice_end_idx] for choice_end_idx in span["choices_end_idx"]])
    answer_embedding = output.last_hidden_state[0][-1]
    return choice_embeddings, answer_embedding



### Choice order invariance achieved!!!

In [183]:
question = "What is the capital of India - please tell me?"
c1 = "New Delhi" 
c2 = "Mumbai"
c3 = "Kolkata"

choice_embeddings, answer_embedding = forward(question, [c1, c2, c3])

choice_embeddings2, answer_embedding2 = forward(question, [c2, c1, c3])

print(torch.allclose(choice_embeddings[0], choice_embeddings2[1], atol=1e-4))
print(torch.allclose(choice_embeddings[1], choice_embeddings2[0], atol=1e-4))
print(torch.allclose(choice_embeddings[2], choice_embeddings2[2], atol=1e-4))

print(torch.allclose(answer_embedding, answer_embedding2, atol=1e-4))



True
True
True
True


### Simple demonstration of a Self Attention Head to process the choice embeddings and answer embedding


![Architecture: the language model reads the state, instruction, choices and answer prompt, and a small head turns the choice and answer embeddings into P(choice)](assets/arch.png)


In [187]:
import torch.nn as nn
import math
class SelfAttention(nn.Module):
    def __init__(self, hidden_dim, new_dim, num_choices):
        super().__init__()
        self.new_dim = new_dim
        self.q = nn.Linear(hidden_dim, new_dim)
        self.k = nn.Linear(hidden_dim, new_dim)
        self.v = nn.Linear(hidden_dim, new_dim)

        self.out_head = nn.Linear(new_dim, num_choices)

    def forward(self, x):

        # x: [B, C+1, H]
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)

        # compute attention weights
        weights = torch.softmax(torch.einsum("bqd,bkd->bqk", [q, k])/math.sqrt(self.new_dim), dim=-1) # [B, C+1, C+1]
        # weights * padding_mask

        out = torch.einsum("bqk,bkd->bqd", [weights, v])

        out = self.out_head(out[:, -1])

        return out
    
self_attn = SelfAttention(hidden_dim=model.config.hidden_size, new_dim=512, num_choices=len(choices))
input_sequence = torch.concat([choice_embeddings, answer_embedding.unsqueeze(0)]).unsqueeze(0)
self_attn(input_sequence)

tensor([[ 0.2428, -0.6077, -1.7079]], grad_fn=<AddmmBackward0>)

### End of this demo | Still TODO (already done in the python files):

- Batching and collating the data
- Implementing the loss function
- Adding Lora Adapters
- Improving the final head
- Training/evaluating the model